In [0]:
%sql 
USE flights_silver;

### Inżynieria i oczyszczanie danych

Dla ułatwienia obliczeń, musimy zmienić kolumny czasowe (*scheduled_departure*, *departure_time*, *wheels_off*, *wheels_on*, *scheduled_arrival*, *arrival_time*) z formatu "hhMM" (string) na timestamp.

Podczas pracy okazało się, że w naszym zestawie danych występują niespójności:
- oznaczenie północy jako "2400" (używane w rekordach lotniczych) zamiast "0000"
 (używane w timpestamp),
- "midnight problem" - jeśli samolot miał planowany wylot blisko północy (np. 00:05), a wyleciał przed czasem (np. 23:50), to podczas zmieniana stringów "hhMM" na timestamp generujemy niepoprawne daty (np. samolot wylatuje 01-01-2015 o 00:05, a przylatuje 01-01-2015 o 23:50, mimo że lot trwał 2h).

Aby uprościć późniejsze wywoływanie kodu, tworzymy funkcję, która:
- tworzy timestamp w kolumnach czasowych
- naprawia oznaczenie północy i dodaje +1 dzień dla przypadków skrajnych  
(np. jeśli lądowanie jest 01-01-2015 o 24:00 to jest to de facto 02-01-2015 o 00:00)

In [0]:
%python
from pyspark.sql.functions import col, when, substring, date_add
def create_fixed_timestamp(df, time_col_name, shift_date_on_2400=False):
    
    # zamiana 2400 na 0000
    time_str_fixed = when(col(time_col_name) == "2400", lit("0000")) \
                     .otherwise(col(time_col_name).cast(StringType()))
    
    # naive timestamp (z poprawionym czasem)
    naive_ts = to_timestamp(
        concat(
            col("year"), lit("-"), col("month"), lit("-"), col("day"), lit(" "),
            substring(time_str_fixed, 1, 2), lit(":"), 
            substring(time_str_fixed, 3, 2), lit(":00")
        ), "y-M-d HH:mm:ss"
    )
    
    # jeśli w pierwotnych danych widniało "2400" dodajemy jeden dzień
    if shift_date_on_2400:
        return when(col(time_col_name) == "2400", date_add(naive_ts, 1)).otherwise(naive_ts)
    else:
        return naive_ts

In [0]:
%python
from pyspark.sql.functions import lit, col, to_date, to_timestamp, concat, expr, when, date_add, date_format
from pyspark.sql.types import IntegerType, StringType

flights_silver = "flights_silver.flights"
df_flights_raw = "flights_bronze.flights"
df = spark.table(df_flights_raw)

max_date = None
# jeśli tabela już istnieje -> wyciągamy maksymalną datę
if spark.catalog.tableExists(flights_silver):
    try:
        # data OSTATNIEGO lotu w tabeli
        row = spark.sql(f"SELECT MAX(scheduled_departure) FROM {flights_silver}").collect()[0]
        max_date = row[0] 
    except:
        max_date = None

df = df.withColumn(
    # musimy zainicjować zmianę stingów na timestamp, żeby mieć z czym porównać max_date
    "scheduled_departure",
    create_fixed_timestamp(df, "scheduled_departure", shift_date_on_2400=True)
)
# jeśli tabela już istnieje -> filtrujemy jedynie nowe dane
if max_date is not None:
    df = df.filter(col("scheduled_departure") > max_date)

# jeśli jest co dopisać
# za pierwszym razem -> full load; potem -> jedynie nowe wiersze 
if df.count() > 0:
    # zaplanowany wylot -> już ogarnięty

    # rzeczywisty wylot
    # tutaj, zamiast konwertować string na timestamp, wyliczamy realny wylot używająć scheduled_departure + departure_delay
    df = df.withColumn(
        "departure_time",
        # make_interval(years, months, days, hours, minutes, seconds, microseconds)
        expr("scheduled_departure + make_interval(0, 0, 0, 0, departure_delay, 0, 0)")
    )

    # planowany przylot -> KONIECZNA zamiana "24:00" na "00:00"
    temp_scheduled_arrival_ts = create_fixed_timestamp(df, "scheduled_arrival", shift_date_on_2400=True)

    # korekta dla lotów nocnych -> jeśli przylot nastąpił wcześniej niż wylot 
    # np.scheduled arrival: 0032; scheduled departure: 2304 
    # arr < dep => shift o 1 dzień ("samolot przylatuje wcześniej niż wylatuje")
    # UWAGA: nie sprawdzi się dla lotów trwających dłużej niż 24h
    df = df.withColumn(
        "scheduled_arrival",
        when(
            col("scheduled_arrival").cast("string") < date_format(col("scheduled_departure"), "HHmm"), # porównujemy stringi HHmm
            date_add(temp_scheduled_arrival_ts, 1)
        ).otherwise(temp_scheduled_arrival_ts)
    )

    # rzeczywisty przylot
    # tutaj, zamiast konwertować string na timestamp, wyliczamy realny wylot używająć scheduled_arrival + arrival_delay
    df = df.withColumn(
        "arrival_time",
        expr("scheduled_arrival + make_interval(0, 0, 0, 0, arrival_delay, 0, 0)")
    )

    # pozostałe kolumny czasowe (wheels_on, wheeels_off)
    for c in ["wheels_off", "wheels_on"]:
        df = df.withColumn(
            c,
            create_fixed_timestamp(df, c, shift_date_on_2400=True)
        )
    
    # obsługa null
    delay_reason_cols = ["air_system_delay", "security_delay", "airline_delay", "late_aircraft_delay", "weather_delay"]
    df = df.fillna(0, subset=delay_reason_cols)

    # finalny zapis
    df_final = df.select(
        col("year"), 
        col("month"), 
        col("day"), 
        col("day_of_week"),
        col("airline"), 
        col("flight_number"), 
        col("tail_number"), 
        col("origin_airport"), 
        col("destination_airport"),
        col("scheduled_departure"), 
        col("departure_time"),
        col("departure_delay"), 
        col("taxi_out"),
        col("wheels_off"),
        col("scheduled_time"),
        col("elapsed_time"),
        col("air_time"),
        col("distance"),
        col("wheels_on"),
        col("taxi_in"),
        col("scheduled_arrival"), 
        col("arrival_time"),      
        col("arrival_delay"), 
        col("diverted"),
        col("cancelled"),
        col("cancellation_reason"),
        col("air_system_delay"),
        col("security_delay"),
        col("airline_delay"),
        col("late_aircraft_delay"),
        col("weather_delay")
    )

    df_final.write.format("delta").mode("append").saveAsTable(flights_silver)

else:
    print("No new records to process.")


In [0]:
%python
display(df_final.limit(5))

year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,departure_time,departure_delay,taxi_out,wheels_off,scheduled_time,elapsed_time,air_time,distance,wheels_on,taxi_in,scheduled_arrival,arrival_time,arrival_delay,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay
2015,1,1,4,AS,98,N407AS,ANC,SEA,2015-01-01T00:05:00.000Z,2014-12-31T13:05:00.000Z,-11,21,2015-01-01T00:15:00.000Z,205,194,169,1448,2015-01-01T04:04:00.000Z,4,2015-01-01T04:30:00.000Z,2014-12-31T06:30:00.000Z,-22,0,0,null,0,0,0,0,0
2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,2015-01-01T00:10:00.000Z,2014-12-31T16:10:00.000Z,-8,12,2015-01-01T00:14:00.000Z,280,279,263,2330,2015-01-01T07:37:00.000Z,4,2015-01-01T07:50:00.000Z,2014-12-31T22:50:00.000Z,-9,0,0,null,0,0,0,0,0
2015,1,1,4,US,840,N171US,SFO,CLT,2015-01-01T00:20:00.000Z,2014-12-31T22:20:00.000Z,-2,16,2015-01-01T00:34:00.000Z,286,293,266,2296,2015-01-01T08:00:00.000Z,11,2015-01-01T08:06:00.000Z,2015-01-01T13:06:00.000Z,5,0,0,null,0,0,0,0,0
2015,1,1,4,AA,258,N3HYAA,LAX,MIA,2015-01-01T00:20:00.000Z,2014-12-31T19:20:00.000Z,-5,15,2015-01-01T00:30:00.000Z,285,281,258,2342,2015-01-01T07:48:00.000Z,8,2015-01-01T08:05:00.000Z,2014-12-31T23:05:00.000Z,-9,0,0,null,0,0,0,0,0
2015,1,1,4,AS,135,N527AS,SEA,ANC,2015-01-01T00:25:00.000Z,2014-12-31T23:25:00.000Z,-1,11,2015-01-01T00:35:00.000Z,235,215,199,1448,2015-01-01T02:54:00.000Z,5,2015-01-01T03:20:00.000Z,2014-12-31T06:20:00.000Z,-21,0,0,null,0,0,0,0,0
